In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)


In [ ]:
# Task 1: Write your code here:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

df_path = os.path.join(path, 'Q1_data.csv')
my_df = pd.read_csv(df_path)

In [ ]:
# Task 2: Write your code here:
my_df.head(10)

In [ ]:
# Task 3: Write your code here:
my_df.info()

In [ ]:
# Task 4: Write your code here:
my_df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(my_df["Delivery_Time"])
plt.title("Delivery Time")

In [ ]:
# Task 1: Write your code here:
my_cleaned_df = my_df.drop("Order_ID", axis = 1).copy()

In [ ]:
# Task 2: Write your code here:
missing_percentage = (my_cleaned_df.isnull().sum() / len(my_cleaned_df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
# Not a lot of data is missing so dropping NA is approprite because data will still be enough
my_cleaned_df1 = my_cleaned_df.dropna()

In [ ]:
# Task 3: Write your code here:
duplicates = my_cleaned_df1.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
# LOTS ??!!!
my_cleaned_df1.drop_duplicates(inplace = True)

In [ ]:
from sklearn.preprocessing import LabelEncoder
# Task 4: Write your code here:
categorical_cols = my_cleaned_df1.select_dtypes(include=["object"]).columns
print(categorical_cols)
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  my_cleaned_df1[col] = le.fit_transform(my_cleaned_df1[col])
  label_encoders[col] = le
  # One-hot is too hard for me oops

In [ ]:
from sklearn.preprocessing import StandardScaler
# Task 5: Write your code here:
scaler = StandardScaler()
my_columns = my_cleaned_df1.drop("Delivery_Time", axis = 1).columns
my_cleaned_df1[my_columns] = scaler.fit_transform(my_cleaned_df1[my_columns])

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(my_cleaned_df1, "Delivery_Time")
# it is so yeah......

In [ ]:
# Task 1: Write your code here:
X = my_cleaned_df1.drop("Delivery_Time", axis=1)
y = my_cleaned_df1["Delivery_Time"]

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import train_test_split , KFold # Not strat cause it is regression.... I think inshallah
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

rf = RandomForestRegressor(
    n_estimators=200, max_depth=20, random_state=42, n_jobs=-1
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
# Without KFold
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X):
  X_train, X_test = X.iloc[train_idx], X.iloc[val_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[val_idx]
  rf.fit(X_train, y_train)
  y_pred = rf.predict(X_test)

  mae_scores.append(mean_absolute_error(y_test, y_pred))
  rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))


mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': my_columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Linear Regression)")
plt.title("Linear Regression: Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: